# 17 · How the beast got its stripes 🌈

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⌛ ~10 min](/lite/notebooks/index.html?path=17-turing-patterns.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/17-turing-patterns.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>

:::{note}
This notebook is **best run locally** — the pattern needs a few thousand small
time steps to grow (a progress bar tells you how far along it is).
:::

In 1952 Alan **Turing** asked a startling question: how does a featureless ball of
cells — an early embryo — decide *where* to put spots, stripes, fingers? His answer
was **reaction–diffusion**. Two substances, an **activator** and an **inhibitor**,
react with each other and diffuse at **different speeds**; that imbalance can make a
perfectly uniform state spontaneously break up into a regular pattern. The same
mechanism is thought to paint **leopard spots, zebra stripes, seashells and fish** —
*morphogenesis*, pattern from no pattern.

So where did **the beast** get its rainbow coat? We put Turing's mechanism on the
**surface of the sculpture itself** — a genuine **PDE on a curved 2-manifold** — and
watch it grow its own skin. Two new ideas at once: **surface finite elements** and a
**coupled nonlinear** system, both written the NGSolve way — **variationally**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import OCCGeometry, Sphere, Cylinder, Glue, Pnt, X, Y, Z
from ngsolve import *
from ngsolve.webgui import Draw
import sys

def progress(i, n):                                    # a tiny dependency-free bar
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:  # (survives JupyterLite/Colab/local)
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  growing the coat… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The beast's skin — a surface mesh

The beast is the **sculpture** from notebook 2 (a spherical shell bored by three
cylinders). Instead of meshing the *solid*, we glue its **faces** into a closed
**shell** and mesh that — a 2-D surface living in 3-D. On such a mesh the measure is
**`ds`** (surface area) and the gradient is the **tangential** gradient
`grad(u).Trace()`; otherwise it is ordinary FEM. We keep the mesh deliberately
**coarse** and resolve the pattern with **higher-order** elements instead
(*p*-refinement): fewer, bigger curved elements, each carrying a quadratic field —
smoother and cheaper than a flood of tiny linear triangles.

In [ ]:
def beast_sculpture():
    s = Sphere(Pnt(50, 50, 50), 80) - Sphere(Pnt(50, 50, 50), 50)
    for p, d in [(Pnt(-100, 0, 0), X), (Pnt(100, -100, 100), Y), (Pnt(0, 100, -100), Z)]:
        s = s - Cylinder(p, d, r=40, h=300)
    return s.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)     # centre + shrink

shell = Glue([f for f in beast_sculpture().faces])               # the closed surface only
mesh = Mesh(OCCGeometry(shell).GenerateMesh(maxh=0.30))           # deliberately coarse
mesh.Curve(2)
print(f"surface mesh: {mesh.nv} vertices, area = {Integrate(CF(1)*ds, mesh):.1f}")
Draw(mesh)

## 2. The model — an activator and an inhibitor

We use the **Gray–Scott** system for two concentrations $u$ and $v$:
$$ \partial_t u = D_u\,\Delta_\Gamma u - u v^2 + F(1-u),\qquad
   \partial_t v = D_v\,\Delta_\Gamma v + u v^2 - (F+k)\,v . $$
The activator $v$ is **autocatalytic** — $uv^2$ turns substrate $u$ into *more* $v$ —
while $F$ feeds fresh $u$ and $k$ removes $v$. The crucial ingredient is
**differential diffusion**: the activator spreads **slower** than the substrate
($D_v<D_u$). That "short-range activation, long-range inhibition" is exactly Turing's
recipe; tune $F,k$ and you get dots ↔ stripes ↔ coral. $\Delta_\Gamma$ is the
**surface** Laplacian.

In [ ]:
fes = H1(mesh, order=2)            # higher order on the coarse mesh (p-refinement)
u, w = fes.TnT()
M = BilinearForm(u * w * ds, check_unused=False).Assemble()                    # surface mass
K = BilinearForm(grad(u).Trace() * grad(w).Trace() * ds, check_unused=False).Assemble()  # surface stiffness

Du, Dv, F, k = 3.6e-3, 1.8e-3, 0.037, 0.060        # "coral" regime; note D_v = D_u/2
dt = 1.0

## 3. A variational semi-implicit (IMEX) scheme

Diffusion is **stiff** (it couples the whole surface), so we take it **implicitly**;
the local **reaction** stays **explicit**. Each species then needs one pre-factorised
solve per step, $M+\Delta t\,D\,K$ — symmetric positive definite, so `sparsecholesky`
fits. The reaction is a **nonlinear form** we never assemble: we just **`Apply`** it
to the current state each step (the same trick as the DG transport in notebook 14) —
no hand-indexing of coefficients, everything stays variational.

In [ ]:
def factor(D):
    mstar = M.mat.CreateMatrix()
    mstar.AsVector().data = M.mat.AsVector() + dt * D * K.mat.AsVector()
    return mstar.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

inv_u, inv_v, M_inv = factor(Du), factor(Dv), M.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

gfu, gfv = GridFunction(fes), GridFunction(fes)
react_u = BilinearForm(fes, nonassemble=True)
react_u += -(-u * gfv * gfv + F * (1 - u)) * w * ds        # r_u(u, v) tested, as an operator
react_v = BilinearForm(fes, nonassemble=True)
react_v += -(gfu * u * u - (F + k) * u) * w * ds           # r_v(u, v); here `u` is the v-field

## 4. Seed the beast and let the pattern grow

We start from a calm skin ($u=1$, $v=0$) and dab a few **patches** of activator onto
the surface. The initial fields are set by an honest **$L^2$-projection** (solve
$M\,\mathbf c = \int (\cdot)\,w\,ds$) — no poking at raw arrays. From those seeds,
spots bud, split and creep across the whole beast: a chemical coral reef.

In [ ]:
def project(cf):                                            # variational L2-projection onto the surface
    g = GridFunction(fes)
    g.vec.data = M_inv * LinearForm(cf * w * ds).Assemble().vec
    return g

seed = sum(IfPos(1.0 - sqrt((x - px)**2 + (y - py)**2 + (z - pz)**2), 1.0, 0.0)
           for (px, py, pz) in [(4, 0, 0), (0, 4, 0), (0, 0, 4),
                                (-4, 0, 0), (0, -4, 0), (0, 0, -4)])
gfu.vec.data = project(1 - 0.5 * seed).vec
gfv.vec.data = project(0.25 * seed).vec

res = gfu.vec.CreateVector()
nsteps = 6500
gfv.AddMultiDimComponent(gfv.vec)                           # frame 0
with TaskManager():
    for step in range(nsteps):
        react_u.Apply(gfu.vec, res); gfu.vec.data = inv_u * (M.mat * gfu.vec - dt * res).Evaluate()
        react_v.Apply(gfv.vec, res); gfv.vec.data = inv_v * (M.mat * gfv.vec - dt * res).Evaluate()
        if step % 260 == 0:
            gfv.AddMultiDimComponent(gfv.vec)               # store a frame
        progress(step, nsteps)

Draw(gfv, mesh, interpolate_multidim=True, animate=True)

Press play: from a few dabs of activator, the beast grows a full coat of **labyrinth
stripes**, entirely on its own — no pattern was ever prescribed, only two reacting,
differently-diffusing chemicals.

:::{dropdown} 🧠 Quiz — why do we need *two different* diffusion rates?
With **equal** rates ($D_u=D_v$) any bump simply smooths out: the uniform state is
stable and **no pattern forms**. Turing's insight is that a **fast-diffusing
inhibitor** and a **slow-diffusing activator** can *destabilise* the uniform state —
a local excess of $v$ reinforces itself (short-range activation) while it is mopped up
at a distance (long-range inhibition). The competition locks in at a characteristic
**wavelength** → spots and stripes. Real coats are believed to use exactly this; so,
now, does our beast.
:::

## Where the ☕ has taken us

Geometry, coefficient functions, spaces, weak forms, solvers, time stepping,
nonlinearity, coupling, even PDEs on curved surfaces — the same handful of ideas,
each notebook a little bolder. And this is only the trailhead: the real **Grand
Expedition** — the rest of the **NGSolve User Meeting** and its fantastic
contributions, and unfitted **CutFEM** with **ngsxfem** — sets off from exactly here.
Pack your coffee. ☕

![The pirate leads a caravan of riders on rainbow mesh tori toward the mountains —
the Grand Expedition.](data/expedition.jpg)